# Gradio UI framework examples
---

In [ ]:
import gradio as gr

from openai import OpenAI

In [ ]:
# Ollama runs everything on this single local endpoint
client = OpenAI(
    base_url="http://localhost:11434/v1/",
    api_key="ollama"  # A placeholder key is required by the client but ignored by Ollama
)

In [ ]:
# set llama
MODEL_LLAMA = "llama3.2"

# set mistral
MODEL_MISTRAL ="mistral"

# set gemma
MODEL_GEMMA = "gemma2:2b"

---

## Simple UI example

In [ ]:
# simple function

def yell(str):
    print(f"Yelling function has been summoned with {str}")
    return str.upper()

yell("Hello")

In [ ]:
# gradio UI for simple function

gr.Interface(
    fn=yell,
    inputs="textbox",
    outputs="textbox",
    flagging_mode="never"
).launch()
# ).launch(inbrowser=True) # if you want the UI to popup in a browser
# ).launch(share=True) # if you want to share this UI with someone via ngrok style sharing into this jupyter notebook
# ).launch(inbrowser=True, auth=("vulz", "potatoes")) # if you want authentication in the UI: auth=(username, password)


---
## One Step Chatbot

In [ ]:
# setup chat function

system_message = "You're a helpful assistant"

def message_llama(prompt):
    messages = [
        {"role":"system","content": system_message},
        {"role": "user", "content": prompt}
    ]

    response = client.chat.completions.create(
        model=MODEL_LLAMA,
        messages=messages
    )

    return response.choices[0].message.content

In [ ]:
message_input = gr.Textbox(label="Your message: ", info="Enter a message for Llama3.2", lines=7)
message_output = gr.Textbox(label='Response: ', lines=8)

view = gr.Interface(
    fn=message_llama,
    title="Llama",
    inputs=[message_input],
    outputs=[message_output],
    examples=["hello", "howdy"],
    flagging_mode="never"
)

view.launch()

### Markdown response (dead space in the left)

In [ ]:
system_message = "You are a helpful assistant that responds in markdown without code blocks"

message_input = gr.Textbox(label="Your message: ", info="Enter a message for Llama3.2", lines=7)
message_output = gr.Markdown(label="Response:")

view = gr.Interface(
    fn=message_llama,
    title="Llama", 
    inputs=[message_input], 
    outputs=[message_output], 
    examples=[
        "Explain the Transformer architecture to a layperson",
        "Explain the Transformer architecture to an aspiring AI engineer",
        ], 
    flagging_mode="never"
    )
view.launch()

# this doesn't show processing or anything like that when the model is working on the answer 
# so the left hand display is dead for sometime before the answer apppears

---

## Streaming One-tep chatbot response

In [ ]:
system_message = "You are a helpful assistant that responds in markdown without code blocks"


def stream_llama(prompt):
    messages = [
        {"role": "system", "content": system_message},
        {"role": "user", "content": prompt}
    ]

    stream = client.chat.completions.create(
        model=MODEL_LLAMA,
        messages=messages,
        stream=True
    )

    result = ""

    for chunk in stream:
        result += chunk.choices[0].delta.content or ""
        yield result

In [ ]:
message_input = gr.Textbox(label="Your message: ", info="Enter a message for Llama3.2", lines=7)
message_output = gr.Markdown(label="Response:")

view = gr.Interface(
    fn=stream_llama,
    title="Llama", 
    inputs=[message_input], 
    outputs=[message_output], 
    examples=[
        "Explain the Transformer architecture to a layperson",
        "Explain the Transformer architecture to an aspiring AI engineer",
        ], 
    flagging_mode="never"
    )
view.launch()

---

## Multi-model Selection Dropdown in Gradio

In [ ]:
# stream llama

system_message = "You are a helpful assistant that responds in markdown without code blocks"


def stream_llama(prompt):
    messages = [
        {"role": "system", "content": system_message},
        {"role": "user", "content": prompt}
    ]

    stream = client.chat.completions.create(
        model=MODEL_LLAMA,
        messages=messages,
        stream=True
    )

    result = ""

    for chunk in stream:
        result += chunk.choices[0].delta.content or ""
        yield result

In [ ]:
# stream mistral

system_message = "You are a helpful assistant that responds in markdown without code blocks"


def stream_mistral(prompt):
    messages = [
        {"role": "system", "content": system_message},
        {"role": "user", "content": prompt}
    ]

    stream = client.chat.completions.create(
        model=MODEL_MISTRAL,
        messages=messages,
        stream=True
    )

    result = ""

    for chunk in stream:
        result += chunk.choices[0].delta.content or ""
        yield result

In [ ]:
# stream gemma

system_message = "You are a helpful assistant that responds in markdown without code blocks"


def stream_gemma(prompt):
    messages = [
        {"role": "system", "content": system_message},
        {"role": "user", "content": prompt}
    ]

    stream = client.chat.completions.create(
        model=MODEL_GEMMA,
        messages=messages,
        stream=True
    )

    result = ""

    for chunk in stream:
        result += chunk.choices[0].delta.content or ""
        yield result

In [ ]:
# the model stream selector function

def stream_model(prompt, model):
    if model == "LLAMA":
        result = stream_llama(prompt)
    elif model == "MISTRAL":
        result = stream_mistral(prompt)
    elif model == "GEMMA":
        result = stream_gemma(prompt)
    else:
        raise ValueError("Unknown Model")

    yield from result

In [ ]:
# the UI

message_input = gr.Textbox(label="Your message: ", info="Enter a message for the LLM:" , lines=7)
model_selector = gr.Dropdown(["LLAMA", "MISTRAL", "GEMMA"], label="Select model", value="LLAMA")
message_output = gr.Markdown(label="Response: ")

view = gr.Interface(
    fn=stream_model,
    title="LLMs",
    inputs=[message_input,model_selector],
    outputs=[message_output],
    examples=[
        ["Explain the Transformer architecture to a layperson", "LLAMA"],
        ["Explain the Transformer architecture to an aspiring AI engineer", "MISTRAL"]
    ],
    flagging_mode="never"
)

view.launch()

---

## Generate Company Brochure in Gradio UI

In [ ]:
system_message = """
You are an assistant that analyzes the contents of a company website landing page
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
"""

In [ ]:
from bs4 import BeautifulSoup
import requests


# Standard headers to fetch a website
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36"
}


def fetch_website_contents(url):
    """
    Return the title and contents of the website at the given url;
    truncate to 2,000 characters as a sensible limit
    """
    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.content, "html.parser")
    title = soup.title.string if soup.title else "No title found"
    if soup.body:
        for irrelevant in soup.body(["script", "style", "img", "input"]):
            irrelevant.decompose()
        text = soup.body.get_text(separator="\n", strip=True)
    else:
        text = ""
    return (title + "\n\n" + text)[:2_000]

In [ ]:
def stream_brochure(company_name, url, model):
    yield ""
    prompt = f"Please generate a company brochure for {company_name}. Here is their landing page:\n"
    prompt += fetch_website_contents(url)
    
    if model == "LLAMA":
        result = stream_llama(prompt)
    elif model == "MISTRAL":
        result = stream_mistral(prompt)
    elif model == "GEMMA":
        result = stream_gemma(prompt)
    else:
        raise ValueError("Unknown Model")
    
    yield from result



In [ ]:
name_input = gr.Textbox(label="Company name:")

url_input = gr.Textbox(label="Landing page URL including http:// or https://")

model_selector = gr.Dropdown(["LLAMA", "MISTRAL", "GEMMA"], label="Select model", value="LLAMA")

message_output = gr.Markdown(label="Response:")

view = gr.Interface(
    fn=stream_brochure,
    title="Brochure Generator", 
    inputs=[name_input, url_input, model_selector], 
    outputs=[message_output], 
    examples=[
        ["Hugging Face", "https://huggingface.co", "LLAMA"],
        ["Edward Donner", "https://edwarddonner.com", "MISTRAL"],
        ["Anthropic", "https://anthropic.com", "GEMMA"]
    ], 
    flagging_mode="never"
    )
view.launch()